# 05 - Orchestrate LakeFlow Job

Programmatically creates a LakeFlow Job with the following DAG:

```
01_Generate_Synthetic_Data
         │
    ┌────┼────┬────┬────┐
    ▼    ▼    ▼    ▼    ▼
  train  train train train train   ← All 5 run in PARALLEL (same notebook, different params)
  alice  bob  carol dave  eve
    │    │    │    │    │
    └────┴────┴────┴────┘
         │
         ▼
03_Deploy_Multi_Model_Endpoint     ← Only after ALL training completes
```

**Key:** The training step uses a single parameterized notebook (`02_Train_User_Model`) 
invoked 5 times with different `user_id` parameters.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import (
    Task,
    NotebookTask,
    Source,
    TaskDependency,
)

w = WorkspaceClient()

# Parameters
dbutils.widgets.text("catalog", "custom_ml", "Catalog")  # Only hardcoded default
dbutils.widgets.text("schema", "hyper_personalization", "Schema")
dbutils.widgets.text("user_ids", "user_alice,user_bob,user_carol,user_dave,user_eve", "User IDs (comma-separated)")
dbutils.widgets.text("endpoint_name", "hyper-personalization-models", "Endpoint Name")
dbutils.widgets.text("job_name", "Hyper Personalization Multi-Model Pipeline", "Job Name")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
USER_IDS = [u.strip() for u in dbutils.widgets.get("user_ids").split(",")]
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
JOB_NAME = dbutils.widgets.get("job_name")

# Notebook paths (derived from current user's workspace)
current_user = w.current_user.me().user_name
BASE_PATH = f"/Workspace/Users/{current_user}/mlops-multi-model-deployment"

print(f"Job Name: {JOB_NAME}")
print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Users: {USER_IDS}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Notebook base path: {BASE_PATH}")

In [0]:
# Build the task DAG
tasks = []

# Task 1: Generate synthetic data
generate_task = Task(
    task_key="generate_data",
    notebook_task=NotebookTask(
        notebook_path=f"{BASE_PATH}/01_Generate_Synthetic_Data",
        source=Source.WORKSPACE,
        base_parameters={
            "catalog": CATALOG,
            "schema": SCHEMA,
            "num_records_per_user": "200",
        },
    ),
)
tasks.append(generate_task)
print(f"Task 1: generate_data -> {generate_task.notebook_task.notebook_path}")

# Tasks 2-N: Train models in parallel (one per user, same notebook)
train_task_keys = []
for user_id in USER_IDS:
    task_key = f"train_{user_id.replace('user_', '')}"
    train_task_keys.append(task_key)
    
    train_task = Task(
        task_key=task_key,
        depends_on=[TaskDependency(task_key="generate_data")],
        notebook_task=NotebookTask(
            notebook_path=f"{BASE_PATH}/02 Train User Model",
            source=Source.WORKSPACE,
            base_parameters={
                "user_id": user_id,
                "catalog": CATALOG,
                "schema": SCHEMA,
            },
        ),
    )
    tasks.append(train_task)
    print(f"Task: {task_key} -> user_id={user_id} (depends on: generate_data)")

# Final task: Deploy endpoint (depends on ALL training tasks)
deploy_task = Task(
    task_key="deploy_endpoint",
    depends_on=[TaskDependency(task_key=tk) for tk in train_task_keys],
    notebook_task=NotebookTask(
        notebook_path=f"{BASE_PATH}/03_Deploy_Multi_Model_Endpoint",
        source=Source.WORKSPACE,
        base_parameters={
            "catalog": CATALOG,
            "schema": SCHEMA,
            "endpoint_name": ENDPOINT_NAME,
            "user_ids": ",".join(USER_IDS),
        },
    ),
)
tasks.append(deploy_task)
print(f"\nTask: deploy_endpoint (depends on: {train_task_keys})")
print(f"\nTotal tasks: {len(tasks)}")

In [0]:
from databricks.sdk.service.jobs import JobSettings

# Create or update the job
try:
    # Check if job with this name already exists
    existing_jobs = [j for j in w.jobs.list(name=JOB_NAME)]
    
    if existing_jobs:
        job_id = existing_jobs[0].job_id
        print(f"Job '{JOB_NAME}' already exists (ID: {job_id}). Updating...")
        w.jobs.reset(
            job_id=job_id,
            new_settings=JobSettings(
                name=JOB_NAME,
                tasks=tasks,
            ),
        )
        print(f"Job updated successfully!")
    else:
        print(f"Creating new job: '{JOB_NAME}'")
        job = w.jobs.create(
            name=JOB_NAME,
            tasks=tasks,
        )
        job_id = job.job_id
        print(f"Job created successfully!")
    
    print(f"\n{'='*60}")
    print(f"Job ID: {job_id}")
    print(f"Job Name: {JOB_NAME}")
    print(f"Tasks: {len(tasks)}")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"Error creating/updating job: {e}")
    raise

In [0]:
import time

# Trigger the job
run = w.jobs.run_now(job_id=job_id)
run_id = run.run_id
print(f"✅ Job run triggered! Run ID: {run_id}")
print(f"   Monitor at: {w.config.host}#job/{job_id}/run/{run_id}")
print(f"\nPolling run {run_id} every 30s...\n")

def poll_run(client, run_id, poll_interval=30):
    """Poll job run until terminal state, printing per-task progress."""
    terminal_states = {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}
    task_states = {}

    while True:
        run_status = client.do("GET", "/api/2.1/jobs/runs/get", query={"run_id": run_id})
        life_cycle = run_status["state"]["life_cycle_state"]

        # Print per-task updates
        for task in run_status.get("tasks", []):
            task_key = task["task_key"]
            task_state = task.get("state", {}).get("life_cycle_state", "PENDING")
            result_state = task.get("state", {}).get("result_state", "-")

            current = f"{task_state} ({result_state})"
            if task_states.get(task_key) != current:
                task_states[task_key] = current
                print(f"  [{task_key}] {task_state} | result: {result_state}")

        # Check if the overall run is done
        if life_cycle in terminal_states:
            result = run_status["state"].get("result_state", "UNKNOWN")
            print(f"\nRun finished: {life_cycle} | Result: {result}")

            # Print duration per task
            print("\nTask durations:")
            for task in run_status.get("tasks", []):
                start = task.get("start_time")
                end = task.get("end_time")
                if start and end:
                    duration_s = (end - start) / 1000
                    print(f"  {task['task_key']}: {duration_s:.1f}s")
            return run_status

        time.sleep(poll_interval)

final_status = poll_run(w.api_client, run_id)